In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("mock-to-star")
    .config("spark.jars.packages", "org.postgresql:postgresql:42.5.4")
    .getOrCreate()
)

PG_URL = "jdbc:postgresql://postgres:5432/bigdata_spark"
PG_PROPS = {
    "user": "postgres",
    "password": "postgres",
    "driver": "org.postgresql.Driver",
}

def pg_read(table: str):
    return spark.read.jdbc(url=PG_URL, table=table, properties=PG_PROPS)

def pg_write(df, table: str, mode: str = "append"):
    df.write.jdbc(url=PG_URL, table=table, mode=mode, properties=PG_PROPS)

mock = pg_read("mock_data")
mock.createOrReplaceTempView("mock_data")
print(f"mock_data rows: {mock.count()}")

## dim_supplier

In [2]:
suppliers = spark.sql("""
    SELECT
        supplier_name          AS name,
        first(supplier_contact) AS contact,
        first(supplier_email)   AS email,
        first(supplier_phone)   AS phone,
        first(supplier_address) AS address,
        first(supplier_city)    AS city,
        first(supplier_country) AS country_name
    FROM mock_data
    WHERE supplier_name IS NOT NULL
    GROUP BY supplier_name
""")

pg_write(suppliers, "dim_supplier")

pg_read("dim_supplier").createOrReplaceTempView("dim_supplier")
print("dim_supplier OK")

dim_supplier OK


## dim_store

In [3]:
stores = spark.sql("""
    SELECT
        store_name             AS store_name,
        first(store_location)  AS location,
        first(store_city)      AS city,
        first(store_state)     AS state,
        first(store_country)   AS country_name,
        first(store_phone)     AS phone,
        first(store_email)     AS email
    FROM mock_data
    WHERE store_name IS NOT NULL
    GROUP BY store_name
""")

pg_write(stores, "dim_store")

pg_read("dim_store").createOrReplaceTempView("dim_store")
print("dim_store OK")

dim_store OK


## dim_product

In [4]:
products = spark.sql("""
    SELECT
        product_name                                AS name,
        first(product_category)                     AS category_name,
        first(product_price)                        AS price,
        first(product_quantity)                     AS quantity,
        first(product_weight)                       AS weight,
        first(product_brand)                        AS brand_name,
        first(product_description)                  AS description,
        first(product_rating)                       AS rating,
        first(product_reviews)                      AS reviews,
        first(product_release_date)                 AS release_date,
        first(product_expiry_date)                  AS expiry_date
    FROM mock_data
    WHERE product_name IS NOT NULL
    GROUP BY product_name
""")

pg_write(products, "dim_product")

pg_read("dim_product").createOrReplaceTempView("dim_product")
print("dim_product OK")

dim_product OK


## dim_customer

In [5]:
customers = spark.sql("""
    SELECT
        customer_first_name        AS first_name,
        first(customer_last_name)  AS last_name,
        first(customer_age)        AS age,
        customer_email             AS email,
        first(customer_country)    AS country_name,
        first(customer_postal_code) AS postal_code
    FROM mock_data
    WHERE customer_email IS NOT NULL
    GROUP BY customer_email, customer_first_name
""")

pg_write(customers, "dim_customer")

pg_read("dim_customer").createOrReplaceTempView("dim_customer")
print("dim_customer OK")

dim_customer OK


## dim_seller

In [6]:
sellers = spark.sql("""
    SELECT
        seller_first_name          AS first_name,
        first(seller_last_name)    AS last_name,
        seller_email               AS email,
        first(seller_country)      AS country_name,
        first(seller_postal_code)  AS postal_code
    FROM mock_data
    WHERE seller_email IS NOT NULL
    GROUP BY seller_email, seller_first_name
""")

pg_write(sellers, "dim_seller")

pg_read("dim_seller").createOrReplaceTempView("dim_seller")
print("dim_seller OK")

dim_seller OK


## fact_sales

In [7]:
for t in ("dim_customer", "dim_seller", "dim_product", "dim_supplier", "dim_store"):
    pg_read(t).createOrReplaceTempView(t)

fact = spark.sql("""
    SELECT
        c.customer_id,
        sel.seller_id,
        p.product_id,
        sup.supplier_id,
        st.store_id,
        m.sale_date                 AS sale_date,
        m.sale_quantity             AS quantity,
        m.sale_total_price          AS total_price
    FROM mock_data m
    JOIN dim_customer  c   ON m.customer_email = c.email
    JOIN dim_seller    sel ON m.seller_email   = sel.email
    JOIN dim_product   p   ON m.product_name   = p.name
    JOIN dim_supplier  sup ON m.supplier_name  = sup.name
    JOIN dim_store     st  ON m.store_name     = st.store_name
""")

pg_write(fact, "fact_sales")
print(f"fact_sales rows written: {fact.count()}")

fact_sales rows written: 10000
